<center><img src="../../image/dice.png" alt="DICE" style="width: 300px;"/></center>

# 13.OpenCV 摄像头色块抓取分拣实验

@－－－－DICE－－－－<br>
@  文件名：14.OpenCV 摄像头色块抓取分拣实验.py <br>
@  版本：V2.0 <br>
@  说明：我们第一步需要开启摄像头作为显示
我们来学习一下摄像头的开启方式之一吧，一般来说，对于我们使用摄像头的时候，首先我们需要测试我们的摄像头是否正常工作，画面是否清晰，让我们一起来测试我们的摄像头吧！！！<br>

我们开始测试摄像头是否可用

# 自适应算法提取色块颜色

### 回到拍照点

In [1]:
! python3 set_arm_pose_by_name.py --pose_name capture_image

I0328 13:21:12.129745 547704799248 set_arm_pose_by_name.py:16] 位姿名称: capture_image
I0328 13:21:12.130181 547704799248 set_arm_pose_by_name.py:18] 机械臂初始化
I0328 13:21:14.266733 547704799248 set_arm_pose_by_name.py:20] 控制机械臂运动到: capture_image


### 运行标定程序，该程序需要在MobaXterm 中运行
python3 annotation_img9points.py

### 在MobaXterm 运行颜色识别代码
python3 color_block_detector.py

# 如果不需要重新提取颜色，代码从这里开始运行

### 增加必要的库

In [1]:
import sys
import math
import time
import cv2
import numpy as np
# 自定义库
sys.path.append("./")
from camera import Camera
from color_block_detector import ColorBlockDetector
from image2workspace import Image2Workspace
from arm5dof_uservo import Arm5DoFUServo
from arm_application import ArmApplication

### 创建显示交互界面

In [2]:
import traitlets
import ipywidgets.widgets as widgets
from IPython.display import display
from jetcam.utils import bgr8_to_jpeg


frame_img = widgets.Image(format='jpeg', width=640, height=480)
display(frame_img)

Image(value=b'', format='jpeg', height='480', width='640')

In [3]:
#####################################
## 物块抓取的配置文件
#####################################
# - 物块像素尺寸最小值
cubic_min_size = 100
# - 物块像素尺寸最大值
cubic_max_size = 300
# - 机械臂抓取位姿偏移量
arm_offset_xy = -10.0 # XY平面上的偏移量
arm_offset_z = 5.0 # Z轴上的偏移量
# - 抓取阈值
# 	在某个位置检测到多少次就进行抓取
cubic_count_threshold = 20
# - 是否只抓取工作区内的物块
# 	True: 只抓取工作区内的物块
#   False: 工作区外的也抓
grab_cubic_in_ws = False
#####################################

# 创建机械臂
arm_config_folder = "config"
arm = Arm5DoFUServo(config_folder=arm_config_folder)
# 运动到图像拍摄点
arm.set_tool_pose(pose_name="capture_image")

In [4]:
# 创建机械臂APP
arm_app = ArmApplication(arm, config_folder=arm_config_folder)
# 要识别的颜色列表
color_name_list = ["red", "green", "blue", "yellow"]
# 创建摄像头
camera = Camera()
# 初始相机
capture = camera.capture
# 载入标定数据
camera.load_cam_calib_data()
# 色块识别器
color_block_detector = ColorBlockDetector()
# 像素到工作台坐标映射器
img2ws = Image2Workspace(camera)
# 机械臂物块抓取
# Key: 位置编码
# Value: (颜色名称, 计数)
candi_workpiece_dict = {} # 候选工件字典
def candi_workpiece_encoding(wx, wy, delta=20):
    '''工件坐标编码, 坐标单位mm'''
    return int(wx/delta) + int(wy/delta)*1000

def justify_workspace_posi(wx, wy, wz):
    '''调整工作空间坐标'''
    theta = math.atan2(wy, wx)
    r = math.sqrt(wx*wx + wy*wy)
    new_r = r + arm_offset_xy
    new_wx = new_r * math.cos(theta)
    new_wy = new_r * math.sin(theta) 
    new_wz = wz + arm_offset_z
    return new_wx, new_wy, new_wz
try:
    while True:
        # 开始时间
        t_start = time.time()
        ret,img_bgr = camera.read()
        if not ret:
            logging.error('图像获取失败')
            break
        # 识别色块
        canvas = np.copy(img_bgr)
        for color_name in color_name_list:
            rect_list, canvas = color_block_detector.color_clock_rect(\
                img_bgr, color_name, canvas=canvas)
            # 遍历所有的ROI框
            for rect in rect_list:
                x, y, w, h = rect
                # 过滤物块
                if w < cubic_min_size or w > cubic_max_size:
                    print(f"像素宽度 = {w}, 不满足尺寸要求")
                    continue
                if h < cubic_min_size or h > cubic_max_size:
                    print(f"像素高度 = {h}, 不满足尺寸要求")
                    continue
                # 计算物体像素中心坐标
                # 计算物体中心坐标
                px = x + w/2
                py = y + h/2
                # 将图像坐标系转换为工作台坐标系
                wx, wy = img2ws.img2workspace(px, py)
                # 判断是否在工作区内
                if grab_cubic_in_ws:
                    if abs(wx) > img2ws.x0 or abs(wy) > img2ws.y0:
                        # 在工作区格子外，不处理
                        print(f"超出工作区范围: wx = {wx} wy = {wy}")
                        continue 
                # 对位置进行编码
                code = candi_workpiece_encoding(wx, wy)
                if code not in candi_workpiece_dict:
                    candi_workpiece_dict[code] = [color_name, 1]
                else:
                    candi_workpiece_dict[code][0] = color_name
                    candi_workpiece_dict[code][1] += 1
                print(f"纳入统计: wx = {wx} wy = {wy} 编码 = {code}")
                if(candi_workpiece_dict[code][1] >= cubic_count_threshold):
                    # 抓取工件
                    print("- 抓取工件")
                    print(f"物块颜色 {color_name} wx={wx:.1f} wy={wy:.1f} mm")
                    wx2, wy2, wz2 = justify_workspace_posi(wx, wy, img2ws.object_height)
                    arm_app.grab_cubic(wx2-11, wy2+10, wz2-10,color_name)
                    # 清空投票字典
                    candi_workpiece_dict = {}
                    break
        # 统计帧率
        t_end = time.time()
        t_pass = t_end - t_start
        fps = int(1/t_pass)

        # 绘制帧率
        cv2.putText(canvas, text=f"FPS:{fps}",\
            org=(20, 20), fontFace=cv2.FONT_HERSHEY_SIMPLEX, \
            fontScale=0.8, thickness=2, lineType=cv2.LINE_AA, color=(0, 0, 255))

        #cv2.imshow("canvas", canvas)
        frame_img.value = bgr8_to_jpeg(canvas)
        key = cv2.waitKey(1)
        if key == ord('q'):
            break
except KeyboardInterrupt:
    print("按键中断")
finally:
    # 关闭所有的窗口
    #cv2.destroyAllWindows()
    # 释放相机
    capture.release()
    # 机械臂卸力
    arm.set_damping(0)

纳入统计: wx = 6.785310582298649 wy = -29.82199919535165 编码 = -1000
纳入统计: wx = 7.168561087294833 wy = -29.887988896148272 编码 = -1000
纳入统计: wx = 7.551811592291017 wy = -29.953978596944893 编码 = -1000
纳入统计: wx = 7.360186339792918 wy = -29.920983746546582 编码 = -1000
纳入统计: wx = 7.558419591554966 wy = -29.821082839772956 编码 = -1000
纳入统计: wx = 7.743436844789102 wy = -29.986973447343203 编码 = -1000
纳入统计: wx = 7.545203593027068 wy = -30.086874354116816 编码 = -1000
纳入统计: wx = 7.360186339792918 wy = -29.920983746546582 编码 = -1000
纳入统计: wx = 7.7500448440530505 wy = -29.854077690171266 编码 = -1000
纳入统计: wx = 7.743436844789102 wy = -29.986973447343203 编码 = -1000
纳入统计: wx = 8.31831260228337 wy = -30.085957998538134 编码 = -1000
纳入统计: wx = 7.935062097287201 wy = -30.019968297741514 编码 = -1000
纳入统计: wx = 7.551811592291017 wy = -29.953978596944893 编码 = -1000
纳入统计: wx = 7.743436844789102 wy = -29.986973447343203 编码 = -1000
纳入统计: wx = 7.928454098023252 wy = -30.152864054913437 编码 = -1000
纳入统计: wx = 7.7434368447891

像素宽度 = 75, 不满足尺寸要求
纳入统计: wx = 10.320469117695623 wy = -28.688261659286127 编码 = -1000
像素宽度 = 74, 不满足尺寸要求
纳入统计: wx = 10.135451864461473 wy = -28.522371051715893 编码 = -1000
像素宽度 = 75, 不满足尺寸要求
像素宽度 = 87, 不满足尺寸要求
像素宽度 = 94, 不满足尺寸要求
纳入统计: wx = 13.08253209614918 wy = -19.78149686203068 编码 = 0
纳入统计: wx = 8.48352603619503 wy = -18.98962045247123 编码 = 0
纳入统计: wx = 7.142149268708394 wy = -18.758656499683056 编码 = 0
纳入统计: wx = 7.717025026202663 wy = -18.857641050877987 编码 = 0
纳入统计: wx = 7.32716652194253 wy = -18.924547107253304 编码 = 0
纳入统计: wx = 7.717025026202663 wy = -18.857641050877987 编码 = 0
纳入统计: wx = 7.89543428017285 wy = -19.156427415620158 编码 = 0
纳入统计: wx = 7.7236330254666115 wy = -18.724745293706064 编码 = 0
纳入统计: wx = 7.888826280908901 wy = -19.289323172792095 编码 = 0
纳入统计: wx = 7.703809027674765 wy = -19.123432565221847 编码 = 0
纳入统计: wx = 7.9086502787007475 wy = -18.890635901276298 编码 = 0
纳入统计: wx = 8.100275531198847 wy = -18.923630751674608 编码 = 0
纳入统计: wx = 7.89543428017285 wy = -19.1564274